# NYC Taxi — Pipeline completo: ETL → Modelos → Streamlit

Este notebook orquesta y valida cada etapa del pipeline distribuido:

```
Bronze (Parquets raw)
    ↓  raw_to_silver.py
Silver (datos limpios, particionados por fleet/month)
    ↓  silver_to_gold.py
Gold  (features ABT + dist_media + agg_hourly)
    ↓  train.py
MLflow Model Registry  (xgb_fare, xgb_duration → Production)
    ↓
Streamlit App  (predicción interactiva)
```

Ejecuta las celdas en orden. Cada sección valida que la etapa anterior haya producido datos correctos antes de continuar.

## 0. Configuración

In [1]:
import os, subprocess, sys
sys.path.insert(0, "/home/jovyan/work/src")

SPARK_MASTER   = os.getenv("SPARK_MASTER", "spark://spark-master:7077")
MLFLOW_URI     = os.getenv("MLFLOW_TRACKING_URI", "http://mlflow:5000")

# Paths inside Spark containers (used for spark-submit jobs)
BRONZE_PATH    = "/opt/spark/data/bronze"
SILVER_PATH    = "/opt/spark/data/silver"
GOLD_PATH      = "/opt/spark/data/gold"

# Paths inside this Jupyter container (used for local file checks)
WORK_DIR       = "/home/jovyan/work"
BRONZE_LOCAL   = f"{WORK_DIR}/data/bronze"
GOLD_LOCAL     = f"{WORK_DIR}/data/gold"

print(f"Spark master : {SPARK_MASTER}")
print(f"MLflow URI   : {MLFLOW_URI}")

Spark master : spark://spark-master:7077
MLflow URI   : http://mlflow:5000


In [2]:
from common.utils import get_spark

spark = get_spark("Pipeline_Notebook", master=SPARK_MASTER)
spark

2026-05-18 15:46:26,416 | INFO | nyc_taxi | SparkSession iniciada: Pipeline_Notebook — master: spark://spark-master:7077


---
## 1. Validar Bronze

Comprueba que los Parquets raw de NYC TLC están disponibles antes de lanzar el ETL.

In [3]:
import glob

parquets = glob.glob(f"{BRONZE_PATH}/*.parquet")
csvs     = glob.glob(f"{BRONZE_PATH}/*.csv")

print("Parquets encontrados:")
for f in sorted(parquets):
    size_mb = os.path.getsize(f) / 1_048_576
    print(f"  {os.path.basename(f):45s}  {size_mb:.1f} MB")

print("\nCSVs encontrados:")
for f in sorted(csvs):
    print(f"  {os.path.basename(f)}")

assert len(parquets) >= 1, "❌ No hay Parquets en data/bronze — descarga los datos NYC TLC primero"
assert any("taxi_zone_lookup" in f for f in csvs), "❌ Falta taxi_zone_lookup.csv en data/bronze"
print("\n✅ Bronze OK")

Parquets encontrados:
  green_tripdata_2024-01.parquet                 1.3 MB
  yellow_tripdata_2024-01.parquet                47.6 MB

CSVs encontrados:
  taxi_zone_lookup.csv
  weather.csv

✅ Bronze OK


---
## 2. ETL: Bronze → Silver

Limpieza, normalización y enriquecimiento con zonas. Registra métricas en MLflow.

In [ ]:
# Lanza raw_to_silver.py via spark-submit desde el contenedor Spark Master.
# Si ya tienes Silver generado puedes saltar esta celda.

result = subprocess.run(
    [
        "docker", "exec", "spark-master",
        "/opt/spark/bin/spark-submit",
        "--master", SPARK_MASTER,
        "/opt/spark/src/etl/raw_to_silver.py",
    ],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("raw_to_silver.py falló")
print("✅ Bronze → Silver completado")

In [ ]:
# Validación Silver
silver = spark.read.parquet(SILVER_PATH)
print(f"Filas Silver : {silver.count():,}")
print(f"Particiones  : {silver.rdd.getNumPartitions()}")
silver.printSchema()
silver.show(5, truncate=False)

In [ ]:
# Distribución por flota y mes
from pyspark.sql import functions as F

silver.groupBy("fleet", "month").count().orderBy("fleet", "month").show()

---
## 3. ETL: Silver → Gold

Feature engineering: genera tabla ABT, dist_media y agg_hourly.

In [ ]:
result = subprocess.run(
    [
        "docker", "exec", "spark-master",
        "/opt/spark/bin/spark-submit",
        "--master", SPARK_MASTER,
        "/opt/spark/src/etl/silver_to_gold.py",
    ],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("silver_to_gold.py falló")
print("✅ Silver → Gold completado")

In [ ]:
# Validación Gold — features ABT
gold_features = spark.read.parquet(f"{GOLD_PATH}/features")
print(f"Gold features : {gold_features.count():,} filas")
gold_features.printSchema()
gold_features.describe("fare_amount", "duration_min", "trip_distance").show()

In [ ]:
# Validación Gold — dist_media (usada por Streamlit)
dist_media = spark.read.parquet(f"{GOLD_PATH}/dist_media")
print(f"Pares PU/DO únicos : {dist_media.count():,}")
dist_media.orderBy(F.col("n_trips").desc()).show(10)

In [ ]:
# Validación Gold — agg_hourly
import plotly.express as px
import pandas as pd

agg = spark.read.parquet(f"{GOLD_PATH}/agg_hourly").toPandas()
fig = px.line(
    agg, x="hour", y="avg_fare", color="fleet",
    title="Tarifa media por hora del día",
    labels={"avg_fare": "Tarifa media ($)", "hour": "Hora"}
)
fig.show()

---
## 4. Entrenamiento de modelos

Entrena `xgb_fare` y `xgb_duration` con GBTRegressor de Spark ML. Los registra en MLflow.

In [ ]:
result = subprocess.run(
    [
        "docker", "exec", "spark-master",
        "/opt/spark/bin/spark-submit",
        "--master", SPARK_MASTER,
        "/opt/spark/src/models/train.py",
    ],
    capture_output=True, text=True
)
print(result.stdout[-4000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("train.py falló")
print("✅ Entrenamiento completado")

In [2]:
# Consulta métricas del último run en MLflow
import mlflow

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.tracking.MlflowClient()

for exp_name in ["nyc_taxi_model"]:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        print(f"Experimento '{exp_name}' no encontrado todavía")
        continue
    runs = client.search_runs(exp.experiment_id, order_by=["start_time DESC"], max_results=1)
    if runs:
        run = runs[0]
        print(f"Último run: {run.info.run_id}")
        print(f"  Status  : {run.info.status}")
        for k, v in run.data.metrics.items():
            print(f"  {k:40s} {v:.4f}")

Último run: 16904fbf3794461a9d60ff5af3cb42ae
  Status  : FINISHED
  xgb_duration_mae                         3.3638
  xgb_duration_r2                          0.7610
  xgb_duration_rmse                        5.7597


---
## 5. Promover modelos a Production

La app Streamlit carga los modelos desde la stage `Production`. 
Esta celda promueve la última versión de cada modelo.

In [3]:
for model_name in ["xgb_fare", "xgb_duration"]:
    versions = client.search_model_versions(f"name='{model_name}'")
    if not versions:
        print(f"⚠️  Modelo '{model_name}' no encontrado en el registry")
        continue
    latest = sorted(versions, key=lambda v: int(v.version), reverse=True)[0]
    client.transition_model_version_stage(
        name=model_name,
        version=latest.version,
        stage="Production",
        archive_existing_versions=True,
    )
    print(f"✅ {model_name} v{latest.version} → Production")


✅ xgb_fare v1 → Production
✅ xgb_duration v1 → Production


/tmp/ipykernel_251/438895991.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.11.3/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


---
## 6. Verificar Streamlit

Antes de lanzar la app, verifica que los recursos que necesita están disponibles.

In [4]:
# Local paths (Jupyter container mounts project at /home/jovyan/work)
checks = {
    "Gold dist_media"      : f"{GOLD_LOCAL}/dist_media",
    "taxi_zone_lookup.csv" : f"{BRONZE_LOCAL}/taxi_zone_lookup.csv",
}

for label, path in checks.items():
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'}  {label:30s}  {path}")

# MLflow model stages
for model_name in ["xgb_fare", "xgb_duration"]:
    versions = client.search_model_versions(f"name='{model_name}'")
    in_prod  = any(v.current_stage == "Production" for v in versions)
    print(f"  {'✅' if in_prod else '❌'}  MLflow model {model_name:20s}  stage=Production")

  ✅  Gold dist_media                 /home/jovyan/work/data/gold/dist_media
  ✅  taxi_zone_lookup.csv            /home/jovyan/work/data/bronze/taxi_zone_lookup.csv
  ✅  MLflow model xgb_fare              stage=Production
  ✅  MLflow model xgb_duration          stage=Production


In [5]:
# Arrancar Streamlit (perfil 'app')
# Ejecutar desde el host (fuera del notebook) o descomenta si prefieres hacerlo aquí:

# result = subprocess.run(
#     ["docker", "compose", "--profile", "app", "up", "-d", "streamlit"],
#     capture_output=True, text=True, cwd="/home/jovyan/work"
# )
# print(result.stdout)

print("Para arrancar Streamlit, ejecuta en el host:")
print("  docker compose --profile app up -d streamlit")
print("  → http://localhost:8501")

Para arrancar Streamlit, ejecuta en el host:
  docker compose --profile app up -d streamlit
  → http://localhost:8501


---
## 7. (Opcional) Structured Streaming

Simula llegada de datos en tiempo real copiando Parquets al hot-folder.

In [ ]:
# Arrancar el streaming job en background
# result = subprocess.run(
#     [
#         "docker", "exec", "-d", "spark-master",
#         "/opt/spark/bin/spark-submit", "--master", SPARK_MASTER,
#         "/opt/spark/src/etl/streaming_job.py",
#     ]
# )

# Simular datos: copiar un Parquet al hot-folder
import shutil, time

SRC = f"{BRONZE_PATH}/yellow_tripdata_2024-01.parquet"
DST = f"/opt/spark/data/streaming_source/batch_{int(time.time())}.parquet"

if os.path.exists(SRC):
    shutil.copy(SRC, DST)
    print(f"✅ Parquet copiado a streaming_source: {os.path.basename(DST)}")
else:
    print(f"⚠️  {SRC} no encontrado — usa otro Parquet de bronze")

---
## Resumen del pipeline

| Etapa | Script | Output | MLflow |
|-------|--------|--------|--------|
| Bronze → Silver | `src/etl/raw_to_silver.py` | `data/silver/` | exp `nyc_taxi_etl` |
| Silver → Gold | `src/etl/silver_to_gold.py` | `data/gold/features/` `data/gold/dist_media/` `data/gold/agg_hourly/` | exp `nyc_taxi_etl` |
| Entrenamiento | `src/models/train.py` | Registry: `xgb_fare`, `xgb_duration` | exp `nyc_taxi_model` |
| Predicción batch | `src/models/predict.py` | stdout / Parquet | — |
| Streaming | `src/etl/streaming_job.py` | `data/gold/streaming/` | — |
| App | `app/app.py` | http://localhost:8501 | carga desde Registry |
